# Hierarchical geolineage model for *Plasmodium falciparum*

This notebook builds a sparse empirical classifier for *P. falciparum* geographic lineages from an Ardal allele matrix and Pf7 sample metadata. The model is trained with leave-one-study-out validation and exported as a schema-validated JSON file for downstream classification.


## Analysis overview

The workflow has four steps:

1. Load the Ardal matrix and metadata table.
2. Define the population hierarchy used by the classifier.
3. Train one local classifier per hierarchy node with leave-one-study-out validation.
4. Export the final model JSON and inspect the build summary.

The notebook assumes the subset matrix files and metadata CSV are available in the working directory.

PLEASE NOTE: This uses a subset of the data used to train Pf geolineage classification models used in Afanc. This subset is the top 100 samples by % callability per geolineage cohort. As such, it will likely differ in its performance.

## Required inputs

| input | expected file | purpose |
|---|---|---|
| allele matrix | `Pf7_subset.bin.zst` | sparse allele presence or absence matrix |
| matrix metadata | `Pf7_subset.bin.meta` | Ardal matrix header metadata |
| sample metadata | `Pf7_subset_meta.csv` | sample, population, and country labels |

The metadata table must contain `Sample`, `Population`, and `Country` columns. The allele ID format passed to Ardal must match the encoded allele headers.


In [1]:
## run once if Ardal is not installed in this kernel
%pip install "git+https://github.com/ArthurVM/Ardal.git"

## general libraries
%pip install --upgrade pandas plotly "nbformat>=5.10.4" scikit-learn zstandard

  Cloning https://github.com/ArthurVM/Ardal.git to /tmp/pip-req-build-yrcefh37
  Running command git clone --filter=blob:none --quiet https://github.com/ArthurVM/Ardal.git /tmp/pip-req-build-yrcefh37
  Resolved https://github.com/ArthurVM/Ardal.git to commit 769af113283f2c1f761bd6fb5defd8d8c801b6f0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.4.6-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached scipy-1.17.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pyjson-1.4.1-py3-none-any.whl.metadata (738 bytes)
  Using cached humanize-4.15.0-py3-none-any.whl.metadata (7.8 kB)
Using cached numpy-2.4.6-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x

In [2]:
from ardal import Ardal
import nbformat  # required by Plotly's notebook renderer
import pandas as pd
import plotly.express as px

from loso_sparse_model import (
    build_geolineage_min_model_sparse,
    write_model_json,
)


## Load inputs

This cell reads the matrix and metadata, then defines the hierarchy used for model training. The `parent_map` links observed population labels to broader geographic branches. Labels that are not listed remain as roots if they pass the sample filters.


In [3]:
## input files
MATRIX_DATA = ["./data/Pf7_subset.bin.zst", "./data/Pf7_subset.bin.meta.gz"]
METADATA_CSV = "./data/Pf7_subset_meta.csv"

## load matrix and metadata
ard_obj = Ardal(
    MATRIX_DATA,
    verbosity="info",
    density_threshold=0.1,
    allele_id_format="{chr}.{start}.{alt}",
)
meta = pd.read_csv(METADATA_CSV, delimiter=",")

## population hierarchy
parent_map = {
    "AF-C": "Africa",
    "AF-E": "Africa",
    "AF-NE": "Africa",
    "AF-W": "Africa",
    "AS-S": "Asia",
    "AS-SE-E": "Asia",
    "AS-SE-W": "Asia",
    "OC-NG": "Oceania",
    "SA": "South America",
}


[2026-06-16 12:31:15,026] INFO: Parsing matrix data from two file paths: ['./data/Pf7_subset.bin.zst', './data/Pf7_subset.bin.meta.gz']
[2026-06-16 12:31:15,027] INFO: Detected matrix format: .bin.zst
[2026-06-16 12:31:17,035] INFO: Initialising Ardal component classes.
[2026-06-16 12:31:17,036] INFO: Decoding headers.
[2026-06-16 12:31:18,011] INFO: Request to compute allele positions with allele_id_format={chr}.{start}.{alt}, bed=None, recompute=False
                                    allele_coords_bed = None
                                    allele_id_format = {chr}.{start}.{alt}
                                    allele_positions_from_bed = False
                                    allele_positions_from_ids = False
[2026-06-16 12:31:24,679] INFO: Initialised HeaderUtils object with 842 GUIDs and 3279295 alleles.
                                    allele_coords_bed = None
                                    allele_id_format = {chr}.{start}.{alt}
                               


--- Ardal Matrix Statistics ---
Number of GUIDs     : 842
Number of Alleles   : 3279295
Allele id format    : {chr}.{start}.{alt}
Number of Sequences : 16
Number of Sites     : 2538820
Matrix Density      : 0.026100014928835926
Roaring Enabled     : True
Bit Matrix Size     : 329.2 MiB
Roaring Matrix Size : 549.8 MiB
Total Matrix Size   : 879.0 MiB
-----------------------------



## Metadata examination

Before training, check the geographic spread and sample balance across geolineages. This is a light diagnostic step to confirm that the metadata labels and coordinates look plausible.


In [4]:
metadata_summary = pd.DataFrame(
    {
        "metric": [
            "samples",
            "geolineages",
            "countries",
            "studies",
            "admin level 1 regions",
        ],
        "value": [
            meta["Sample"].nunique(),
            meta["Population"].nunique(),
            meta["Country"].nunique(),
            meta["Study"].nunique(),
            meta["Admin level 1"].nunique(),
        ],
    }
)
metadata_summary


,metric,value
0,samples,842
1,geolineages,9
2,countries,27
3,studies,47
4,admin level 1 regions,54


In [5]:
countries_by_geolineage = (
    meta.groupby("Population")
    .agg(
        sample_count=("Sample", "nunique"),
        country_count=("Country", "nunique"),
        countries=("Country", lambda values: ", ".join(sorted(set(values)))),
    )
    .sort_values(["country_count", "sample_count"], ascending=False)
    .reset_index()
)

countries_by_geolineage


,Population,sample_count,country_count,countries
0,AF-W,100,10,"Benin, Cameroon, Côte d'Ivoire, Gabon, Gambia,..."
1,AS-SE-E,100,4,"Cambodia, Laos, Thailand, Vietnam"
2,AF-NE,93,4,"Ethiopia, Kenya, Sudan, Uganda"
3,AF-E,100,3,"Kenya, Malawi, Tanzania"
4,AS-SE-W,100,2,"Myanmar, Thailand"
5,OC-NG,100,2,"Indonesia, Papua New Guinea"
6,SA,49,2,"Colombia, Peru"
7,AF-C,100,1,Democratic Republic of the Congo
8,AS-S,100,1,Bangladesh


In [6]:
sample_map_df = meta.copy()
sample_map_df["map_latitude"] = sample_map_df["Admin level 1 latitude"].fillna(
    sample_map_df["Country latitude"]
)
sample_map_df["map_longitude"] = sample_map_df["Admin level 1 longitude"].fillna(
    sample_map_df["Country longitude"]
)

fig = px.scatter_geo(
    sample_map_df,
    lat="map_latitude",
    lon="map_longitude",
    color="Population",
    hover_name="Sample",
    hover_data={
        "Country": True,
        "Admin level 1": True,
        "Study": True,
        "Year": True,
        "map_latitude": False,
        "map_longitude": False,
    },
    projection="natural earth",
    title="Pf7 subset samples coloured by geolineage",
    opacity=0.75,
)
fig.update_traces(marker={"size": 6, "line": {"width": 0.3, "color": "white"}})
fig.update_layout(
    legend_title_text="geolineage",
    margin={"l": 0, "r": 0, "t": 48, "b": 0},
)
fig.show()


## Model settings

The settings below prioritise a compact lineage panel while requiring each retained marker to be supported across held-out folds. The default scoring metric is macro F1, which gives each lineage equal weight during model selection.

Key tuning parameters:

| parameter | role |
|---|---|
| `top_k_per_lineage` | candidate markers retained per lineage before co-occurrence pruning |
| `max_model_alleles` | upper bound on markers evaluated per local classifier |
| `min_lineage_samples` | minimum samples required for a lineage to enter training |
| `min_fold_support` | fraction of folds in which a marker must recur |
| `cooc_threshold` | removes highly co-occurring markers within the candidate pool |
| `min_stable_alleles_per_lineage` | target minimum number of stable markers per lineage where possible |


In [ ]:
OUTPUT_JSON = "pf_geolineage_example_model.json"

MODEL_PARAMETERS = {
    "ard_obj": ard_obj,
    "meta_df": meta,
    "parent_map": parent_map,
    "sample_col": "Sample",
    "lineage_col": "Population",
    "study_col": "Country",
    "method": "kullbackleibler",
    "top_k_per_lineage": 200,
    "max_model_alleles": 500,
    "min_lineage_samples": 5,
    "min_fold_support": 0.5,
    "smoothing_alpha": 1.0,
    "uniform_priors": True,
    "threads": 16,
    "metric": "macro_f1",
    "species_id": "Plasmodium_falciparum",
    "model_id": "pf_geolineage_empirical_model",
    "cooc_threshold": 0.9,
    "score_tolerance": 0.005,
    "min_stable_alleles_per_lineage": 5,
}


## Build and export the model

This is the computationally expensive step. Each country is held out in turn, and each hierarchy node receives a local classifier over its direct children. The final model is then refit using the stable allele set selected from the cross-validation folds.


In [ ]:
final_model, fold_results_by_node = build_geolineage_min_model_sparse(**MODEL_PARAMETERS)

write_model_json(final_model, OUTPUT_JSON)


## Review build summary

The exported JSON file is the deployable model. The summary gives the final panel size and hierarchy metadata. The fold counts by node are useful for checking whether each local classifier had enough validation support.


In [ ]:
final_model["summary"]
